In [1]:
import dask
import dask.distributed
from dask_util import DaskClient
import numpy as np

In [2]:
local_params = {
    "n_workers":4, 
    "processes" : True, 
    "dashboard_address" : 'localhost:7777'
    
}

client = DaskClient(local_params=local_params)

In [3]:
client.getWorkerIds()

['tcp://127.0.0.1:33741',
 'tcp://127.0.0.1:34825',
 'tcp://127.0.0.1:36981',
 'tcp://127.0.0.1:39039']

In [4]:
%load_ext autoreload
%autoreload 2
import SepVector
import __pyDaskVector as DaskVector
import Hypercube
import pyOperator as Op


WARNING! DATAPATH not found. The folder /tmp will be used to write binary files


In [5]:

ns = [10,10]
os = [0,0]
ds = [1,1]
chunks = (1,3)

ax = Hypercube.axis(n=1, o=1, d=1)
hyp = Hypercube.hypercube(ns=ns, ds=ds, os=os)
vec = SepVector.getSepVector(ns=ns, ds=ds, os=os, storage='dataFloat')
vec.set(1)

floatVector
Axis 1: n=10	o=0.000000	d=1.000000
Axis 2: n=10	o=0.000000	d=1.000000

In [6]:
# option 1
# creating from scratch
data = DaskVector.DaskVector(client, vecCls=SepVector.floatVector, ns=ns, ds=ds, os=os, chunks=chunks)

In [7]:
# option 2
# creating from existing in-memory SepVector
daskVec = DaskVector.DaskVector(client, from_vector=vec, chunks=chunks)

In [8]:
client.getClient().has_what()

{'tcp://127.0.0.1:33741': ('floatVector-8f069c4b305deda924a50774c26baf57',
  'floatVector-3bcc7b69bc3780f2b8785a6ea3f68a05',
  'floatVector-e9772f2ced35917ce11a8cbc2911dec9',
  'floatVector-1e47a56f2c31db0274f50a422f035962',
  'window-c4ce2fdfa4be079990254c8ab9296002',
  'window-8611ac130e68a91ef97a90fb9d49eabc',
  'window-f13eec6e4b36b3765b747f2c4ced3fc7'),
 'tcp://127.0.0.1:34825': ('floatVector-b10758d2258af6199f4574b34d646757',
  'floatVector-75001f04de98ad55879a14893b470d20',
  'floatVector-0ef36c3d7228a8005d0dd81734607a59',
  'floatVector-1f8d6d4a1ce3cf00889cc1ad7e7eb598',
  'window-b1616aaab3c60f2af72619b780f3e7a6',
  'window-80cd207a1bdd22155dccb55ceec0a418',
  'window-e8acb7206a3c691c4332e29ba5f06b16',
  'window-768f1706cb2b912a94dcab7b22e91196'),
 'tcp://127.0.0.1:36981': ('floatVector-f37104722fcf442d1f829f1ad2ddc490',
  'floatVector-7b0db7d448f12ce9f967e42f045303be',
  'floatVector-0a723e0940dbdc09ba757302959aaca8',
  'floatVector-8ebf8846b3680b0daa4eadf56d76414f',
  'window-13df14dd64c0e70f3d0cc11fd93d7e88',
  'window-5a0182c180b6551f6c8d2d6b46da081a',
  'window-49cbdfeaecc3a824a1d1d0ce3950483f',
  'window-6f631597ed2d5e781356a04cc346a7a5'),
 'tcp://127.0.0.1:39039': ('floatVector-6a36cad1d5a3521a3db902a203966ba2',
  'floatVector-c705817811e74c160cd688f5e130c755',
  'floatVector-11bb92a93e9b82c47cd5ee0f51230a01',
  'window-880539910ca51f10d6e2ad62ed824456',
  'window-9f4068cd2360f00893e05c1b4c840ff0',
  'window-a92788d0b547066359739278a56e121b',
  'window-4a0557b2bc05e9d57e95b0794d8e93be')}

In [9]:
# daskVec[3:5,:] = 0

In [10]:
daskVec[:]

[array([[1., 1.],
        [1., 1.],
        [1., 1.]], dtype=float32),
 array([[1., 1.],
        [1., 1.],
        [1., 1.]], dtype=float32),
 array([[1., 1.],
        [1., 1.],
        [1., 1.]], dtype=float32),
 array([[1., 1.],
        [1., 1.],
        [1., 1.]], dtype=float32),
 array([[1., 1.],
        [1., 1.],
        [1., 1.]], dtype=float32),
 array([[1., 1.],
        [1., 1.],
        [1., 1.]], dtype=float32),
 array([[1., 1.],
        [1., 1.],
        [1., 1.]], dtype=float32),
 array([[1., 1.],
        [1., 1.],
        [1., 1.]], dtype=float32),
 array([[1., 1.],
        [1., 1.],
        [1., 1.]], dtype=float32),
 array([[1., 1.],
        [1., 1.],
        [1., 1.]], dtype=float32),
 array([[1., 1.],
        [1., 1.],
        [1., 1.],
        [1., 1.]], dtype=float32),
 array([[1., 1.],
        [1., 1.],
        [1., 1.],
        [1., 1.]], dtype=float32),
 array([[1., 1.],
        [1., 1.],
        [1., 1.],
        [1., 1.]], dtype=float32),
 array([[1., 1.],
     

In [11]:

# zeroOp = DaskVector.DaskOperator(client, opCls=Op.ZeroOp, domain=daskVec, range=daskVec)

In [12]:
# zeroOp.forward(False, daskVec, daskVec)

In [13]:
scaleOp = DaskVector.DaskOperator(client, opCls=Op.scalingOp, domain=daskVec, range=data, op_args=[4])

In [14]:

# for _ in range(10):
#     scaleOp.forward(False, daskVec, data)
#     err = []
#     for v1, v2 in zip(data[:], daskVec[:]):
#         err.append(np.linalg.norm(v1-4*v2))
#     print(err)

In [15]:
data[:]

[array([[0., 0.],
        [0., 0.],
        [0., 0.]], dtype=float32),
 array([[0., 0.],
        [0., 0.],
        [0., 0.]], dtype=float32),
 array([[0., 0.],
        [0., 0.],
        [0., 0.]], dtype=float32),
 array([[0., 0.],
        [0., 0.],
        [0., 0.]], dtype=float32),
 array([[0., 0.],
        [0., 0.],
        [0., 0.]], dtype=float32),
 array([[0., 0.],
        [0., 0.],
        [0., 0.]], dtype=float32),
 array([[0., 0.],
        [0., 0.],
        [0., 0.]], dtype=float32),
 array([[0., 0.],
        [0., 0.],
        [0., 0.]], dtype=float32),
 array([[0., 0.],
        [0., 0.],
        [0., 0.]], dtype=float32),
 array([[0., 0.],
        [0., 0.],
        [0., 0.]], dtype=float32),
 array([[0., 0.],
        [0., 0.],
        [0., 0.],
        [0., 0.]], dtype=float32),
 array([[0., 0.],
        [0., 0.],
        [0., 0.],
        [0., 0.]], dtype=float32),
 array([[0., 0.],
        [0., 0.],
        [0., 0.],
        [0., 0.]], dtype=float32),
 array([[0., 0.],
     

In [18]:
d2= scaleOp.forward(True, daskVec, data)

In [19]:
data[:]

[array([[8., 8.],
        [8., 8.],
        [8., 8.]], dtype=float32),
 array([[8., 8.],
        [8., 8.],
        [8., 8.]], dtype=float32),
 array([[8., 8.],
        [8., 8.],
        [8., 8.]], dtype=float32),
 array([[8., 8.],
        [8., 8.],
        [8., 8.]], dtype=float32),
 array([[8., 8.],
        [8., 8.],
        [8., 8.]], dtype=float32),
 array([[8., 8.],
        [8., 8.],
        [8., 8.]], dtype=float32),
 array([[8., 8.],
        [8., 8.],
        [8., 8.]], dtype=float32),
 array([[8., 8.],
        [8., 8.],
        [8., 8.]], dtype=float32),
 array([[8., 8.],
        [8., 8.],
        [8., 8.]], dtype=float32),
 array([[8., 8.],
        [8., 8.],
        [8., 8.]], dtype=float32),
 array([[8., 8.],
        [8., 8.],
        [8., 8.],
        [8., 8.]], dtype=float32),
 array([[8., 8.],
        [8., 8.],
        [8., 8.],
        [8., 8.]], dtype=float32),
 array([[8., 8.],
        [8., 8.],
        [8., 8.],
        [8., 8.]], dtype=float32),
 array([[8., 8.],
     